# Cross integration with `multibench`

**Cross integration** has several batches in which **all** modalities are present; the task is removing batch effects while keeping biological structure. Spatial registration (PASTE, PASTE2, SPIRAL, GPSA) also lives under `cross` - see the note at the end.

This tutorial covers, end to end:

- installing the package and the per-method environments
- the on-disk data layout this category expects
- seeing what runs on a dataset (`scan`) and what each method exposes for tuning
- running one method live, then a whole benchmark sweep with metrics
- reading the two metric families and drawing the standard figures
- **running the same pipeline on your own dataset**, demonstrated for real

**Reference dataset:** `D52` (23,478 cells). The stored results shipped
with these notebooks were produced on it, so every table below reproduces.

## Install

Prerequisites: Linux, `conda` (mamba recommended) and ~230 GB free disk
during the build. From the repository directory:

```bash
pip install -e .                # the multibench package + CLI
multibench env doctor           # which method environments exist / are missing
multibench env install --run    # build them all from the committed lockfiles
```

Each method runs in its **own conda environment** (they need mutually
incompatible framework versions), so the wrapper can run torch 1.x, torch 2.x,
TensorFlow and R methods in one sweep. `env install` is a dry run until you add
`--run`.

Measured on a clean machine: **29 environments, ~50 min build, 175 GB** (plus a
52 GB package cache you can drop afterwards with `conda clean -a`). Details and
the smallest end-to-end check live in `SETUP.md`.

In [ ]:
import warnings; warnings.filterwarnings("ignore")
%matplotlib inline
from pathlib import Path
import pandas as pd
pd.set_option("display.max_colwidth", None)   # never truncate a `reason`
pd.set_option("display.max_columns", None)    # never hide a metric column
pd.set_option("display.width", 200)
import multibench as mtb

RESULTS = Path("results")     # stored sweep results, so comparisons reproduce
print("multibench", mtb.__version__)

In [ ]:
DATASET  = "D52"
CATEGORY = "cross"

## 1. The data layout

A dataset is a **folder of flat files**; the folder name is the dataset name.
`describe_layout` prints the exact filenames for each category:

In [ ]:
print(mtb.describe_layout(CATEGORY))

The modality files are HDF5 with three required datasets:

| dataset | contents | shape |
|---|---|---|
| `matrix/data` | the matrix, **features x cells** | `(n_features, n_cells)` |
| `matrix/features` | one name per feature | `(n_features,)` |
| `matrix/barcodes` | one id per cell | `(n_cells,)` |

Note this is the **transpose** of the scanpy/AnnData convention (`AnnData.X` is
cells x genes). Two safety nets exist: `mtb.io.to_canonical(src, dst)` converts
an `.h5ad` correctly, and `scan()` rejects a transposed file at preflight instead
of letting a method fail half an hour in.

**The label CSV** is the one file you author by hand, so its schema in full:
one row per cell, in **the same order as `matrix/barcodes`** of the matching
modality file(s); the cell-type label is the last column (or a column named
`x`); a header row is expected. Where a category uses several label files
(`cty1.csv`, `rna_cty.csv`, ...), each aligns with its own batch or modality.
The next cell prints the head of a shipped one - this is the whole format:

In [ ]:
cty = sorted((mtb.config.DEFAULT.data_path / DATASET).glob("*cty*.csv"))[0]
print(cty.name)
print(*open(cty).read().splitlines()[:4], sep="\n")

> **ATAC caution.** Methods disagree about the ATAC representation - some need
> **gene-activity scores**, others need **peaks** - and feeding the wrong one
> runs to completion and returns a plausible but wrong embedding, with no error.
> `describe_layout` above states which file resolves where; check what your
> files actually contain before trusting a result.

**From AnnData to canonical, executed.** Most real data starts as `.h5ad`;
`mtb.io.to_canonical` writes the layout above correctly (including the
transpose). Converting a small demo object end to end:

In [ ]:
import anndata as ad, numpy as np, h5py, tempfile, os
tmp = tempfile.mkdtemp()
demo = ad.AnnData(X=np.random.poisson(2.0, size=(120, 40)).astype(float))
demo.obs_names = [f"cell{i}" for i in range(120)]
demo.var_names = [f"gene{i}" for i in range(40)]
src = os.path.join(tmp, "demo.h5ad"); demo.write_h5ad(src)

dst = os.path.join(tmp, "rna.h5")
mtb.io.to_canonical(src, dst)
with h5py.File(dst) as f:
    print("keys :", sorted(f["matrix"].keys()))
    print("shape:", f["matrix/data"].shape, "(features x cells - transposed for you)")

## 2. What can I run on this dataset?

`scan` inspects the folder and reports every method that can run - and, for the
rest, exactly why not (missing file, missing environment, wrong layout). Nothing
executes, so this is instant and safe.

In [ ]:
avail = mtb.scan(DATASET, category=CATEGORY)
avail[avail.runnable][["method", "modalities", "env", "output_kind",
                       "n_tunable", "runtime_tier"]]

Methods that are *not* runnable come with a reason rather than a silent absence:

In [ ]:
not_ok = avail[~avail.runnable][["method", "modalities", "reason"]]
not_ok.head(5) if len(not_ok) else "(everything in this category runs here)"


## 2b. How much of the paper does this cover?

`scan()` answers "what runs on THIS dataset". A different question: how many of
the methods the paper benchmarks for **cross** does this package wire at all?
Stated explicitly so you never mistake a dataset limitation for full coverage.

In [ ]:
from multibench.engine import registry

PAPER = {'vertical': ['totalVI', 'sciPENN', 'Concerto', 'scMSI', 'Matilda', 'MOFA2', 'Multigrate', 'UINMF', 'scMoMaT', 'Seurat_WNN', 'scMM', 'scMDC', 'moETM', 'VIMCCA', 'iPOLNG', 'MIRA', 'UnitedNet', 'scMVP'], 'diagonal': ['scBridge', 'Portal', 'SCALEX', 'VIPCCA', 'Seurat_v3', 'MultiMAP', 'Seurat_v5', 'sciCAN', 'Conos', 'iNMF', 'online_iNMF', 'scJoint', 'GLUE', 'uniPort'], 'mosaic': ['MultiVI', 'scMoMaT', 'StabMap', 'Cobolt', 'UINMF', 'Multigrate', 'SMILE', 'scMM', 'moETM', 'UnitedNet', 'totalVI', 'sciPENN'], 'cross': ['totalVI', 'scMoMaT', 'UnitedNet', 'sciPENN', 'Concerto', 'scMDC', 'StabMap', 'UINMF', 'scMM', 'MOFA2', 'Multigrate', 'PASTE', 'PASTE2', 'SPIRAL', 'GPSA']}
IMPUTATION_ONLY = ['scMM', 'moETM', 'UnitedNet', 'totalVI', 'sciPENN']

paper = PAPER[CATEGORY]
wired = sorted({m for m in mtb.list_methods()
                if any(v.when.get("category") == CATEGORY
                       for v in registry.get(m).variants)})
missing = [m for m in paper if m not in wired]
print(f"paper benchmarks {len(paper)} methods for {CATEGORY}; this package wires {len(wired)}")
if missing:
    print("not wired here:", ", ".join(missing))
    imp = [m for m in missing if m in IMPUTATION_ONLY]
    if imp:
        print("  the paper evaluates these only via IMPUTATION, which is not wired:",
              ", ".join(imp))
else:
    print("full parity with the paper for this category")

## 3. What can I tune?

`params_for` reports each method's defaults and, where the upstream script
exposes any, the tunable hyperparameters. **An empty `tunable` is honest**: many
upstream scripts hardcode their hyperparameters, and this package never edits
upstream code, so it reports rather than pretends.

In [ ]:
rows = []
for m in avail[avail.runnable]["method"]:
    try:
        p = mtb.params_for(m, CATEGORY)
    except Exception:                      # multi-variant: needs modalities
        mods = avail[avail.method == m].iloc[0]["modalities"].split("+")
        p = mtb.params_for(m, CATEGORY, mods)
    rows.append({"method": m, "n_tunable": len(p.get("tunable") or {}),
                 "tunable": ", ".join(sorted((p.get("tunable") or {}))[:6])})
pd.DataFrame(rows).sort_values("n_tunable", ascending=False).reset_index(drop=True)

## 4. Run one method

`run_all` runs methods end to end: resolve inputs -> run in the method's own
conda env -> load the output -> compute metrics -> keep everything in a
`BatchResult`. StabMap is among the fastest cross methods - `run_sec` below is the measured time on our host.

In [ ]:
res = mtb.run_all("D52", CATEGORY,
                  methods=["StabMap"],
                  out_dir="/tmp/tutorial_cross")
res.summary

(The lines scrolling above the table are scIB's own progress chatter from
the metric computation - harmless; the result is the summary row below.)

The metrics are already computed - `run_all` picked the right label files,
resolved the label order (see `label_order` in the summary), and scored the
embedding. One figure:

In [ ]:
res.plot()

## 5. Run the whole benchmark

The same call without `methods=` runs everything runnable. On `D52` that is
hours of compute, so this cell is shown but not executed here - the stored
results it produced are loaded right below.

```python
res = mtb.run_all(DATASET, CATEGORY, out_dir="results/cross_all",
                  timeout=4*3600,       # a hung method is recorded, not fatal
                  skip_existing=True)   # resume instead of repeating hours
print(res.failures)                     # ALWAYS check: failures are recorded, not raised
```

`run_all` writes `summary.csv`, `long.csv` and `failures.csv` into `out_dir` -
the stored files under `results/` loaded below are exactly those, kept so the
notebook reproduces without re-running the sweep. (The live demo above used
reduced settings where noted; the stored sweep ran defaults, so its `run_sec`
differs.)

Three behaviours worth knowing before a long sweep:

- **a method that fails is a row, not an exception** - the sweep continues and
  `res.failures` carries the error text;
- **`timeout=` bounds each method's whole step** including metric computation;
- **`skip_existing=True` resumes** a killed sweep, but refuses to combine with
  `params=` (it cannot know the old output used your new parameters).

In [ ]:
summary = pd.read_csv(RESULTS / "summary_D52.csv")
cols = [c for c in ["method","status","run_sec",
                    "ARI","NMI","ASW","iASW","iF1","cLISI",      # clustering
                    "ASW_batch","GC","iLISI"                     # batch correction
                    ] if c in summary.columns]
summary[cols]

## 6. Reading the metrics

Two families, matching the paper's grouping. All are **higher = better**, on
[0, 1] except ARI (can be slightly negative at chance level).

| family | metrics | what they measure |
|---|---|---|
| clustering / bio-conservation | `ARI`, `NMI`, `ASW`, `iASW`, `iF1`, `cLISI` | does the embedding separate the annotated cell types? |
| batch correction | `ASW_batch`, `GC`, `iLISI` (+ opt-in `kBET`) | are the batches mixed within each cell type? |

Notes that save confusion later:

- `iASW`/`iF1` are **isolated-label** scores; this benchmark scores *every*
  label, so they exist even on a single-batch dataset.
- batch metrics appear only when the dataset has real batches - their absence on
  a single-batch dataset is correct, not missing data.
- `label_order` in the summary records which label files scored the embedding,
  in which order - when several candidate orders fit the cell counts, each is
  screened and the best kept; a low `label_order_confidence` means the ranking
  was close and `label_order_candidates` in the record shows the alternatives.
- `kBET` is opt-in (`mtb.evaluate(..., slow_metrics=True)`): it spawns R per
  method and costs hours per dataset.
- graph-output methods come in two kinds: scMoMaT also ships a secondary UMAP
  embedding, which is what gets scored (status `CHAIN_OK_GRAPH_METHOD`), while
  Seurat_WNN emits only the graph - it legitimately has **no** embedding metrics
  and its status says so rather than scoring garbage.

## 7. The figures

**Per-dataset bubble chart** - radius encodes rank (rank 1 = largest bubble),
colour the value:

In [ ]:
long = pd.read_csv(RESULTS / "long_all_D52.csv")
fig = mtb.plot.bubble(long, title="cross integration - D52")
fig.set_dpi(110)
fig

**Across datasets** - when you have `long` tables from several datasets,
concatenate them and `mtb.plot.bar` summarises each method across all of them
(only methods present in more than one dataset are truly comparative):

```python
allx = pd.concat([long_d1, long_d2, ...], ignore_index=True)
mtb.plot.bar(allx, group="clustering")   # or group="batch", or all metrics
```

## 8. Your own dataset - for real

Everything above used shipped data. This section does what you will actually do:
put files in a folder, point the package at it, and get scored results - executed
here on a dataset the package has never seen (a 60% cell subsample of
`D52` under a new name, built with ordinary h5py/pandas code you can
adapt to your own export pipeline).

In [ ]:
import os, shutil
import h5py
import numpy as np
import pandas as pd

def subsample_dataset(src_dir, dst_dir, frac=0.6, seed=0):
    """Copy a dataset to a new name, keeping a random fraction of the cells.

    Files sharing a cell count get the SAME kept-cell index, so modality files
    and their label CSVs stay aligned - which is exactly the property your own
    export pipeline must preserve. The output is the canonical layout:
    matrix/data as features x cells, plus matrix/features and matrix/barcodes.
    """
    rng = np.random.default_rng(seed)
    os.makedirs(dst_dir, exist_ok=True)
    counts, keep = {}, {}
    for fn in sorted(os.listdir(src_dir)):
        p = os.path.join(src_dir, fn)
        if fn.endswith(".h5"):
            with h5py.File(p) as f:
                if "matrix/data" in f:
                    counts[fn] = f["matrix/data"].shape[1]   # features x cells
        elif fn.endswith(".csv"):
            counts[fn] = len(pd.read_csv(p))
    for n in set(counts.values()):
        k = max(50, int(n * frac))
        keep[n] = np.sort(rng.choice(n, size=k, replace=False))
    for fn, n in counts.items():
        sp, dp = os.path.join(src_dir, fn), os.path.join(dst_dir, fn)
        idx = keep[n]
        if fn.endswith(".csv"):
            pd.read_csv(sp).iloc[idx].to_csv(dp, index=False)
        else:
            with h5py.File(sp) as f, h5py.File(dp, "w") as g:
                grp = g.create_group("matrix")
                grp.create_dataset("data", data=np.asarray(f["matrix/data"])[:, idx])
                if "matrix/features" in f:
                    grp.create_dataset("features", data=np.asarray(f["matrix/features"]))
                if "matrix/barcodes" in f:
                    grp.create_dataset("barcodes", data=np.asarray(f["matrix/barcodes"])[idx])
    return dst_dir

In [ ]:
DATA_ROOT = "/tmp/mydata"
src = mtb.config.DEFAULT.data_path / "D52"
subsample_dataset(src, f"{DATA_ROOT}/MYDATA_cross", frac=0.6)

sc = mtb.scan(f"MYDATA_cross", category=CATEGORY, data_path=DATA_ROOT)
print(f"{int(sc.runnable.sum())} of {len(sc)} methods can run on MYDATA_cross")

In [ ]:
mine = mtb.run_all(f"MYDATA_cross", CATEGORY,
                   methods=["StabMap"],
                   out_dir=f"{DATA_ROOT}/out_cross",
                   data_path=DATA_ROOT)
mine.summary

In [ ]:
mine.plot()

StabMap reached CHAIN_OK on this subsample when we ran it; `run_sec` above is the measured time.

For your real data the only work is producing the canonical files: export each
modality with `mtb.io.to_canonical` (from `.h5ad`) or the h5py pattern above,
write one label CSV per the layout in section 1, and the same three calls -
`scan`, `run_all`, `plot` - do the rest.

## Troubleshooting

| symptom | meaning | fix |
|---|---|---|
| `scan` says not runnable: input files not found | a required file is absent | the reason names the exact file and lists what IS in the folder |
| `scan` says env missing | that method's conda env is not built | `multibench env install --run` |
| `... looks like cells x features` | matrix stored transposed | re-export with `mtb.io.to_canonical` |
| a method FAILs in seconds | wrong input representation or layout | read `res.failures.iloc[0]["error"]` - the full command line and stderr tail are there |
| a method TIMEOUTs | slow, not broken | raise `timeout=`; runtime tiers in `scan` are measured, not guessed |
| `label_order_confidence` low | several label files fit the cell count | check `label_order_candidates` in the record |

### Note - spatial registration

`PASTE`, `PASTE2`, `SPIRAL` and `GPSA` are cross-integration methods whose output
is **aligned spatial coordinates**, not an embedding - their status reports
`RUN_OK_NO_EMBEDDING` and clustering metrics genuinely do not apply. Point them
at a directory of spatial slices (see `mtb.scan("D63", category="cross")`).

## Next steps

- the other three tutorials: **vertical**, **diagonal**, **mosaic**
- `SETUP.md` - measured install cost and the smallest end-to-end check
- `mtb.method_info(name)` - everything the registry knows about one method
- `mtb.sweep(...)` - one method over a range of one hyperparameter